# OneIE GENEVA — curated-v2 baseline

Fresh training on the shared frozen splits. Keep old results as legacy experiments.
Default **check** mode does not train. Review data checks, then explicitly select **smoke** and confirm.
After smoke completes, select **full** in a separate run with its saved smoke directory available.
DEV full-gold argument classification F1 selects the checkpoint; TEST is evaluated afterward.
This variant uses mBERT, no global features, and unrestricted event-role combinations.
The common scorer matches argument fragments by event type, span and role; it does not evaluate discontinuous grouping.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys, zipfile

DATASET = 'GENEVA'
# Kaggle may automatically extract uploaded ZIP files.
manifests = list(Path('/kaggle/input').rglob('bundle-manifest.json'))
bundles = list(Path('/kaggle/input').rglob('oneie-curated-v2.zip'))
assert len(manifests) == 1 or len(bundles) == 1, 'Attach the curated-v2 dataset'
ROOT = manifests[0].parent if len(manifests) == 1 else Path('/kaggle/working/oneie-curated-v2')
BUNDLE = bundles[0] if bundles else None
RUN_MODE = 'check'  # 'check', 'smoke', or 'full'
CONFIRM_RUN = False
INSTALL_DEPENDENCIES = False
SMOKE_RUN = Path('/kaggle/working/runs/' + DATASET + '-smoke')
FULL_RUN = Path('/kaggle/working/runs/' + DATASET + '-full')
assert RUN_MODE in ('check', 'smoke', 'full')


In [ ]:
# Extract only a local attached bundle. This does not contact Kaggle APIs.
if not ROOT.exists():
    assert BUNDLE.is_file(), f'Attach the prepared ZIP and set BUNDLE: {BUNDLE}'
    with zipfile.ZipFile(BUNDLE) as archive:
        names = archive.namelist()
        assert len(names) == len(set(names)), 'Duplicate ZIP paths'
        for name in names:
            target = (ROOT / name).resolve()
            assert target.is_relative_to(ROOT.resolve()), f'Unsafe ZIP path: {name}'
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
assert (ROOT / 'bundle-manifest.json').is_file()
# Verify bytes before importing any bundled code.
import hashlib
manifest = json.loads((ROOT / 'bundle-manifest.json').read_text())
for name, expected in manifest['files'].items():
    path = (ROOT / name).resolve()
    assert path.is_relative_to(ROOT.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name
patch_info = json.loads((ROOT / 'OneIE/.curated-patch.json').read_text())
assert patch_info['patch_version'] == 2, 'Attach dataset version with source patch 2'
print('Bundle checksums OK; OneIE source patch 2')


In [ ]:
# Optional dependency installation; restart the kernel if the environment requests it.
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(ROOT / 'configs/oneie/requirements.txt')], check=True)
env = dict(os.environ, PYTHONHASHSEED='42', TOKENIZERS_PARALLELISM='false', PYTHONUNBUFFERED='1')
def run(*args):
    subprocess.run([sys.executable, '-m', *map(str, args)], cwd=ROOT, env=env, check=True)
run('scripts.bundle_oneie', '--verify', ROOT)


In [ ]:
base = ['scripts.run_oneie_dataset', '--prepared', ROOT / 'prepared' / DATASET,
        '--oneie', ROOT / 'OneIE']
# Default behavior: validate inputs without allocating a model.
run(*base, '--output', FULL_RUN)


In [ ]:
if RUN_MODE == 'check':
    print('Checks finished. No training started.')
else:
    assert CONFIRM_RUN, 'Review checks, then explicitly set CONFIRM_RUN=True'
    if RUN_MODE == 'smoke':
        run(*base, '--output', SMOKE_RUN, '--execute', '--smoke')
    else:
        # Full training refuses an absent, incomplete or mismatched smoke run.
        run(*base, '--output', FULL_RUN, '--execute', '--smoke-run', SMOKE_RUN,
            '--epochs', 30, '--seed', 42, '--micro-batch', 1, '--accumulate', 8)


In [ ]:
output = SMOKE_RUN if RUN_MODE == 'smoke' else FULL_RUN
if (output / 'scores.json').exists():
    print((output / 'scores.json').read_text())
    print('Save the entire run directory, including run.json, completed.json, best.pt and predictions.')
